In [ ]:
#!/usr/bin/env python3
"""
NB08: Spatiotemporal Graph Attention Network (ST-GNN)
Replaces ConvLSTM with a proper graph-based spatiotemporal model.
Each grid cell is a node; edges connect 8-neighbors.
Uses multi-head graph attention (GAT) implemented in pure PyTorch.
"""



# NB 08 — ST-GNN Spatiotemporal Forecasting

Replaces the ConvLSTM approach with a **Spatial Graph Attention Network (GAT)**.
Each 49×49 grid cell is a graph node; edges connect all 8-neighbor cells.
Node features = [last 5 daily mission counts, day-of-week, month].
The GAT learns spatially-varying attention weights, revealing which
neighboring areas most influence each cell's demand.

**Expected runtime: ~5-10 minutes (CPU); ~2 min (MPS/CUDA)**


In [ ]:
Setup
import json, sys, warnings
warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# Load config
PROJECT_ROOT = Path('.')
V3_DIR   = PROJECT_ROOT / 'mda_project' / 'data' / 'processed_v3'
OUT_DIR  = PROJECT_ROOT / 'mda_project' / 'data' / 'output'
FIG_DIR  = OUT_DIR / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device('mps' if torch.backends.mps.is_available() else
                      'cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")




In [ ]:
Load data and build grid
mission = pd.read_parquet(V3_DIR / 'mission_records_v3.parquet')
m = mission.dropna(subset=['t0']).copy()
m['t0'] = pd.to_datetime(m['t0'])
m['date']    = m['t0'].dt.date
m['dow']     = m['t0'].dt.dayofweek    # 0=Mon
m['month']   = m['t0'].dt.month

LAT_BINS = np.linspace(49.5, 51.6, 50)  # 49 cells
LON_BINS = np.linspace(2.5,  6.5,  50)

dates = sorted(m['date'].unique())
N_DAYS = len(dates)
H, W   = 49, 49
N      = H * W  # 2401 graph nodes

print(f"Building {N_DAYS}-day × {H}×{W} grid...")
grids = np.zeros((N_DAYS, H, W), dtype=np.float32)
day_meta = []
for di, d in enumerate(dates):
    day = m[m['date'] == d]
    g, _, _ = np.histogram2d(day['latitude'], day['longitude'],
                              bins=[LAT_BINS, LON_BINS])
    grids[di] = g.astype(np.float32)
    day_meta.append({'date': d, 'dow': day['dow'].iloc[0],
                     'month': day['month'].iloc[0]})

# Normalize grid counts per day by global max
GRID_MAX = grids.max() + 1e-6
grids_norm = grids / GRID_MAX
print(f"Grid stats: max={grids.max():.0f}, mean={grids.mean():.3f}, "
      f"nonzero/day={(grids>0).sum(axis=(1,2)).mean():.0f}")




In [ ]:
Build graph adjacency (8-neighbor sparse structure)
# edges: list of (src, dst) pairs for 8-connected grid
print("Building 8-neighbor graph adjacency...")
rows_e, cols_e = [], []
for r in range(H):
    for c in range(W):
        src = r * W + c
        for dr in [-1, 0, 1]:
            for dc in [-1, 0, 1]:
                if dr == 0 and dc == 0:
                    continue
                nr, nc = r + dr, c + dc
                if 0 <= nr < H and 0 <= nc < W:
                    dst = nr * W + nc
                    rows_e.append(src)
                    cols_e.append(dst)

edge_index = torch.tensor([rows_e, cols_e], dtype=torch.long)
n_edges = edge_index.shape[1]
print(f"Graph: {N} nodes, {n_edges} edges "
      f"(avg degree: {n_edges/N:.1f})")




In [ ]:
Build sequence dataset
# For each day t, input = [grid_t-5..t-1, dow_t, month_t], target = grid_t
SEQ_LEN = 5
day_meta_df = pd.DataFrame(day_meta)

def make_sequences(grids_n, meta_df, seq_len):
    X_spatial, X_temporal, Y = [], [], []
    for t in range(seq_len, len(grids_n)):
        # Spatial: last seq_len grids flattened to (N, seq_len) node features
        seq = grids_n[t-seq_len:t]   # (seq_len, H, W)
        spatial = seq.reshape(seq_len, -1).T  # (N, seq_len)
        X_spatial.append(spatial)
        # Temporal: global context (same for all nodes on this day)
        dow   = meta_df.iloc[t]['dow'] / 6.0
        month = (meta_df.iloc[t]['month'] - 1) / 11.0
        X_temporal.append([dow, month])
        # Target
        Y.append(grids_n[t].reshape(-1))  # (N,)
    return (np.stack(X_spatial).astype(np.float32),
            np.array(X_temporal, dtype=np.float32),
            np.stack(Y).astype(np.float32))

Xs, Xt, Y = make_sequences(grids_norm, day_meta_df, SEQ_LEN)
print(f"Dataset: {len(Y)} samples x {N} nodes (Xs:{Xs.shape}, Xt:{Xt.shape})")

# Train/val/test split: 60/20/20 on temporal order
n = len(Y)
n_tr = int(n * 0.60); n_va = int(n * 0.80)
print(f"Train: {n_tr}, Val: {n_va-n_tr}, Test: {n-n_va}")

def to_tensor(*arrs): return [torch.tensor(a) for a in arrs]

Xs_tr, Xt_tr, Y_tr = to_tensor(Xs[:n_tr], Xt[:n_tr], Y[:n_tr])
Xs_va, Xt_va, Y_va = to_tensor(Xs[n_tr:n_va], Xt[n_tr:n_va], Y[n_tr:n_va])
Xs_te, Xt_te, Y_te = to_tensor(Xs[n_va:], Xt[n_va:], Y[n_va:])

BS = 16
train_dl = DataLoader(TensorDataset(Xs_tr, Xt_tr, Y_tr), batch_size=BS, shuffle=True)
val_dl   = DataLoader(TensorDataset(Xs_va, Xt_va, Y_va), batch_size=BS)
test_dl  = DataLoader(TensorDataset(Xs_te, Xt_te, Y_te), batch_size=BS)




In [ ]:
ST-GNN Architecture: GAT + Temporal Encoding
class GraphAttentionLayer(nn.Module):
    """Single-head Graph Attention Layer (Veličković et al. 2018)."""
    def __init__(self, in_dim, out_dim, dropout=0.1):
        super().__init__()
        self.W  = nn.Linear(in_dim, out_dim, bias=False)
        self.a  = nn.Linear(2 * out_dim, 1, bias=False)
        self.drop = nn.Dropout(dropout)
        self.leaky = nn.LeakyReLU(0.2)

    def forward(self, x, edge_index):
        # x: (N, in_dim), edge_index: (2, E)
        h = self.W(x)  # (N, out_dim)
        src, dst = edge_index[0], edge_index[1]
        e = self.leaky(self.a(torch.cat([h[src], h[dst]], dim=-1)))  # (E,1)
        # Softmax over neighbors per node (sparse)
        alpha = torch.zeros(h.shape[0], device=h.device)
        # Manual scatter softmax
        exp_e = torch.exp(e.squeeze() - e.squeeze().max())  # numerical stability
        alpha_sum = torch.zeros(h.shape[0], device=h.device)
        alpha_sum.scatter_add_(0, dst, exp_e)
        alpha_norm = exp_e / (alpha_sum[dst] + 1e-10)
        alpha_norm = self.drop(alpha_norm)
        # Aggregate
        out = torch.zeros_like(h)
        out.scatter_add_(0, dst.unsqueeze(-1).expand_as(h[src]),
                         alpha_norm.unsqueeze(-1) * h[src])
        return torch.relu(out), alpha_norm  # return attention weights

class STGNN(nn.Module):
    """2-layer ST-GAT with temporal context injection."""
    def __init__(self, seq_len=5, temp_dim=2, hidden=32, heads=4):
        super().__init__()
        self.input_proj = nn.Linear(seq_len, hidden)
        self.temp_proj  = nn.Linear(temp_dim, hidden)
        # Multi-head GAT: concat outputs of 'heads' single-head layers
        self.gat1_heads = nn.ModuleList(
            [GraphAttentionLayer(hidden, hidden//heads) for _ in range(heads)])
        self.gat2      = GraphAttentionLayer(hidden, hidden)
        self.out_proj  = nn.Linear(hidden, 1)
        self.drop      = nn.Dropout(0.15)

    def forward(self, xs, xt, edge_index):
        # xs: (B, N, seq_len), xt: (B, 2)
        B, N, SL = xs.shape
        # Project spatial history
        h = torch.relu(self.input_proj(xs.view(B*N, SL)))  # (B*N, hidden)
        # Inject temporal context (broadcast over nodes)
        temp = torch.relu(self.temp_proj(xt))  # (B, hidden)
        temp_exp = temp.unsqueeze(1).expand(B, N, -1).reshape(B*N, -1)
        h = h + temp_exp
        h = self.drop(h)

        # Process each sample separately through GAT
        out = torch.zeros(B, N, device=xs.device)
        attn_weights = []
        for b in range(B):
            hb = h[b*N:(b+1)*N]  # (N, hidden)
            # Multi-head GAT layer 1
            heads_out = []
            for gat_head in self.gat1_heads:
                ho, aw = gat_head(hb, edge_index)
                heads_out.append(ho)
                if b == 0: attn_weights.append(aw.detach().cpu())
            hb = torch.cat(heads_out, dim=-1)  # (N, hidden)
            hb = self.drop(hb)
            # GAT layer 2
            hb, _ = self.gat2(hb, edge_index)
            out[b] = self.out_proj(hb).squeeze(-1)
        return torch.relu(out), attn_weights




In [ ]:
Train ST-GNN
print("Training ST-GNN (2-layer GAT, 4 heads)...")
model = STGNN(seq_len=SEQ_LEN, hidden=32, heads=4).to(device)
ei = edge_index.to(device)
opt = optim.Adam(model.parameters(), lr=3e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=40)
crit = nn.L1Loss()

EPOCHS = 40
tr_loss, va_loss = [], []
best_val, best_state = float('inf'), None

for epoch in range(EPOCHS):
    model.train(); bl = []
    for xs, xt_b, y in train_dl:
        xs, xt_b, y = xs.to(device), xt_b.to(device), y.to(device)
        opt.zero_grad()
        pred, _ = model(xs, xt_b, ei)
        loss = crit(pred, y)
        loss.backward(); opt.step()
        bl.append(loss.item())
    tr_loss.append(np.mean(bl))

    model.eval(); vb = []
    with torch.no_grad():
        for xs, xt_b, y in val_dl:
            xs, xt_b, y = xs.to(device), xt_b.to(device), y.to(device)
            pred, _ = model(xs, xt_b, ei)
            vb.append(crit(pred, y).item())
    va_loss.append(np.mean(vb))
    scheduler.step()
    if va_loss[-1] < best_val:
        best_val = va_loss[-1]
        best_state = {k:v.clone() for k,v in model.state_dict().items()}
    if (epoch+1) % 10 == 0:
        print(f"  Epoch {epoch+1:3d}: train={tr_loss[-1]:.5f}, val={va_loss[-1]:.5f}")

model.load_state_dict(best_state)




In [ ]:
Evaluate: ST-GNN vs baselines
model.eval()
preds_te, trues_te, attn_te = [], [], []
with torch.no_grad():
    for xs, xt_b, y in test_dl:
        xs, xt_b, y = xs.to(device), xt_b.to(device), y.to(device)
        pred, aw = model(xs, xt_b, ei)
        preds_te.append(pred.cpu().numpy())
        trues_te.append(y.cpu().numpy())
        if not attn_te: attn_te = aw

preds_te = np.concatenate(preds_te)  # (n_test, N)
trues_te = np.concatenate(trues_te)

# Persistence baseline: predict today = yesterday
Xs_te_np = Xs_te.numpy()  # (n_test, N, SEQ_LEN)
persist   = Xs_te_np[:, :, -1]  # last known day

mae_sgnn    = np.mean(np.abs(preds_te - trues_te)) * GRID_MAX
mae_persist = np.mean(np.abs(persist  - trues_te)) * GRID_MAX
improvement = (mae_persist - mae_sgnn) / mae_persist * 100

# Convert to actual counts
preds_counts = preds_te * GRID_MAX
trues_counts = trues_te * GRID_MAX

print(f"\n  Persistence MAE:  {mae_persist:.4f} events/cell/day")
print(f"  ST-GNN MAE:       {mae_sgnn:.4f} events/cell/day")
print(f"  Improvement:      {improvement:.1f}% over persistence")




In [ ]:
Save model comparison CSV
mc_ext = pd.DataFrame([
    {'Model':'Persistence', 'Task':'Grid Forecast',
     'MAE_events_per_cell': round(mae_persist,4), 'Improvement_%': 0},
    {'Model':'ST-GNN (GAT)', 'Task':'Grid Forecast',
     'MAE_events_per_cell': round(mae_sgnn,4),
     'Improvement_%': round(improvement,1)},
])
mc_ext.to_csv(FIG_DIR / 'table_08_stgnn_evaluation.csv', index=False)




In [ ]:
Attention weight spatial analysis
# Aggregate attention weights → mean attention per destination node
attn_mean = torch.stack(attn_te[:1]).mean(0).numpy() if attn_te else np.zeros(n_edges)
attn_by_node = np.zeros(N)
for i, dst in enumerate(cols_e):
    if i < len(attn_mean):
        attn_by_node[dst] += attn_mean[i] if i < len(attn_mean) else 0
attn_grid = attn_by_node.reshape(H, W)




In [ ]:
Publication figure: Fig 5
print("Rendering Fig 5 (ST-GNN results)...")
fig5, axes = plt.subplots(2, 2, figsize=(14, 10), dpi=200)
fig5.suptitle('NB 08 — ST-GNN Spatiotemporal Graph Attention Network',
              fontsize=15, fontweight='bold', y=0.98)

# (a) Training convergence
ax = axes[0, 0]
ax.plot(range(1, EPOCHS+1), [v*GRID_MAX for v in tr_loss],
        color='#1a5276', lw=2, label='Train MAE')
ax.plot(range(1, EPOCHS+1), [v*GRID_MAX for v in va_loss],
        color='#e67e22', lw=2, label='Val MAE')
ax.axvline(np.argmin(va_loss)+1, color='green', ls='--', alpha=0.7, label='Best val epoch')
ax.set_xlabel('Epoch'); ax.set_ylabel('MAE [events/cell/day]')
ax.set_title('(a) ST-GNN Training Convergence', fontweight='bold')
ax.legend(); ax.grid(True, ls=':', alpha=0.5)

# (b) Model comparison bar chart
ax = axes[0, 1]
models_c = ['Persistence', 'ST-GNN (GAT)']
maes_c   = [mae_persist, mae_sgnn]
colors_c = ['#a93226', '#1a5276']
bars = ax.bar(models_c, maes_c, color=colors_c, edgecolor='white', width=0.5)
for bar, v in zip(bars, maes_c):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
            f'{v:.4f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.set_ylabel('Test MAE [events/cell/day]')
ax.set_title('(b) Model Comparison: ST-GNN vs Baseline', fontweight='bold')
ax.text(0.5, 0.85, f'ST-GNN improves {improvement:.1f}%\nover persistence baseline',
        transform=ax.transAxes, ha='center', fontsize=11,
        bbox=dict(facecolor='#d5e8d4', edgecolor='#82b366', boxstyle='round,pad=0.4'))
ax.grid(True, ls=':', alpha=0.5, axis='y')

# (c) Prediction vs truth scatter (sample one test day)
ax = axes[1, 0]
sample_true  = trues_counts[0].flatten()
sample_pred  = preds_counts[0].flatten()
# Only plot cells with > 0 true value
mask = sample_true > 0
ax.scatter(sample_true[mask], sample_pred[mask], alpha=0.4, s=10, color='#2471a3')
lim = max(sample_true.max(), sample_pred.max()) * 1.05
ax.plot([0, lim], [0, lim], 'r--', lw=1.5, label='Perfect prediction')
ax.set_xlabel('True counts [events/cell]')
ax.set_ylabel('Predicted counts [events/cell]')
ax.set_title('(c) Prediction vs Truth (one test day)', fontweight='bold')
ax.legend(); ax.grid(True, ls=':', alpha=0.5)

# (d) Spatial attention heatmap
ax = axes[1, 1]
im = ax.imshow(attn_grid, cmap='hot', interpolation='nearest', aspect='auto')
plt.colorbar(im, ax=ax, shrink=0.8, label='Mean attention received')
ax.set_title('(d) GAT Spatial Attention Map\n(heat = most attended nodes)',
             fontweight='bold')
ax.set_xlabel(f'Longitude bins (2.5°→6.5° E)')
ax.set_ylabel(f'Latitude bins (49.5°→51.6° N)')

plt.tight_layout(rect=[0, 0, 1, 0.96])
fig5.savefig(FIG_DIR / 'fig5_stgnn_results.png', dpi=200,
             bbox_inches='tight', facecolor='white')
plt.close(fig5)
print(f"  -> saved fig5_stgnn_results.png")

print("\n=== NB08 Complete ===")
print(f"  ST-GNN MAE: {mae_sgnn:.4f} | Persistence MAE: {mae_persist:.4f} | "
      f"Improvement: {improvement:.1f}%")

